# CHAID（SPSS寄せ）Notebook

このノートは「まず自動で型判定→結果確認→必要な箇所だけ手動上書き→再実行」を最短で回すためのものです。

- レアカテゴリを `Other` にまとめる処理は **入れていません**（要求どおり）。
- 代わりに、木の“形”を制御するパラメータ（深さ・最小ノードサイズ・有意水準・連続値のビン数など）を **Notebook上で即調整**できるようにしています。

> 重要：結果を見ながらパラメータを変えるのは探索として有効ですが、やり過ぎると「都合の良い木」になりやすいので、目的（説明/施策/予測）に合わせて触る範囲を決めるのが安全です。


In [ ]:
# === 0) imports ===
from chaid_spss_fast_bars_image_v6 import CHAIDTree
import pandas as pd
import numpy as np
import os
import shutil
import inspect
import re
from datetime import datetime
from IPython.display import Image, display


# 画面がうるさくなる FutureWarning を非表示（分析結果には影響しません）
import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r"Downcasting object dtype arrays on \.fillna.*",
)


## 1) データ読み込み

`dummy_data.csv` をあなたのデータに置き換えてください。

In [ ]:
# === 1) load data ===
df_raw = pd.read_csv("train.csv")
df = df_raw.copy()
df.head()


## 1.5) 説明変数の除外設定（任意）

- `user_id` や `session_id` のように **ほぼユニークなID列**が入っていると、木がそれに引っ張られて解釈不能になりがちです。
- ここでは **(A) 手動除外** と **(B) 自動検出（IDっぽい列の除外）** を用意しています。
- 自動検出は保守的（=誤除外しにくい）にしてあります。必要なら閾値を調整してください。


In [ ]:
# === 1.5) exclude predictors (optional) ===

# (A) 手動で除外したい説明変数（ここを編集）
MANUAL_EXCLUDE_COLS = [
    # 例: "user_id",
    # 例: "session_id",
]

# (B) 自動で「IDっぽい列」を除外するか
AUTO_EXCLUDE_ID_LIKE = True

# 自動検出のルール（必要なら調整）
# - 列名がIDっぽい（*_id, id, uuid など） + ほぼユニーク なら除外
# - 文字列列で、ほぼユニークなら除外（例: UUID文字列）
# - 整数列で、ほぼユニークなら除外（例: user_id が int の場合）
AUTO_EXCLUDE_NAME_PATTERNS = [
    r"(^id$)|(_id$)",
    r"uuid",
    r"guid",
    r"user.?id",
    r"session.?id",
]
AUTO_EXCLUDE_UNIQUE_RATIO_THRESHOLD = 0.98  # 非欠損のうち何割がユニークなら除外するか
AUTO_EXCLUDE_MIN_NUNIQUE = 50               # ユニーク数がこれ未満なら除外しない（誤除外防止）

print("Manual exclude:", MANUAL_EXCLUDE_COLS)
print("Auto exclude:", AUTO_EXCLUDE_ID_LIKE, "| unique_ratio>=", AUTO_EXCLUDE_UNIQUE_RATIO_THRESHOLD)


## 1.6) レアカテゴリを `Other` にまとめる（任意：割合ベース）

- `30件以下` のような固定件数ではなく、**全体の N% 未満**のカテゴリを `Other` にまとめるためのオプションです。
- 既定では **OFF**（`OTHERIZE_ENABLED=False`）。必要なときだけONにしてください。
- `Other` の中身が分からなくなる問題を避けるため、**どのカテゴリをOther化したかの一覧（mapping）を表示**します。

注意：この処理は **見やすさのための前処理**なので、施策や解釈に使う場合は「Other化したカテゴリ一覧」を必ず一緒に保存・共有するのがおすすめです。


In [ ]:
# === 1.6) optional: collapse rare categories into 'Other' by % threshold ===

# ONにすると、指定した列の「全体のN%未満」のカテゴリを Other にまとめます
OTHERIZE_ENABLED = True
OTHER_MIN_PCT = 0.01   # 例: 0.01 = 1%
OTHER_COLS = [
    # 例: "Job",
]
OTHER_LABEL = "Other"

# (任意) 重み列がある場合は列名を指定（例: "weight"）。無ければ None のままでOK
OTHER_WEIGHT_COL = None

# 欠損（NaN）はOtherにせず、そのまま残す（推奨）
KEEP_MISSING_SEPARATE = True

# (任意) Other化したカテゴリ一覧をJSONに保存したい場合
SAVE_OTHER_MAPPING = False
OTHER_MAPPING_PATH = "other_mapping.json"

def _otherize_column(df_in, col, min_pct, other_label="Other", weight_col=None, keep_missing=True):
    s = df_in[col]
    # counts と pct を作る（重みがあれば重み付き）
    if weight_col and weight_col in df_in.columns:
        w = df_in[weight_col].fillna(0)
        tmp = pd.DataFrame({col: s, "_w": w})
        counts = tmp.groupby(col, dropna=not keep_missing)["_w"].sum()
    else:
        counts = s.value_counts(dropna=not keep_missing)
    total = float(counts.sum())
    pct = (counts / total) if total != 0 else counts * 0.0

    rare = pct[pct < float(min_pct)].index.tolist()
    # NaNはOther化しない（keep_missing=Trueのとき）
    if keep_missing:
        rare = [x for x in rare if pd.notna(x)]
    rare_set = set(rare)

    def _map(x):
        if pd.isna(x):
            return x
        return other_label if x in rare_set else x

    s2 = s.map(_map)
    return s2, counts, pct, rare

# このセルを実行すると毎回 df を df_raw から作り直す（再実行でブレないように）
df = df_raw.copy()

other_mapping = {}
report_rows = []

if OTHERIZE_ENABLED and len(OTHER_COLS) > 0:
    for col in OTHER_COLS:
        if col not in df.columns:
            report_rows.append({"col": col, "status": "skip (not found)"})
            continue
        s2, counts, pct, rare = _otherize_column(
            df, col, OTHER_MIN_PCT, other_label=OTHER_LABEL,
            weight_col=OTHER_WEIGHT_COL, keep_missing=KEEP_MISSING_SEPARATE
        )
        df[col] = s2
        other_mapping[col] = [str(x) for x in rare]

        # まとめ：Otherに入ったカテゴリ数など
        report_rows.append({
            "col": col,
            "min_pct": float(OTHER_MIN_PCT),
            "other_label": OTHER_LABEL,
            "n_categories": int((counts.index.notna()).sum()),
            "n_otherized": int(len(rare)),
        })

    report_df = pd.DataFrame(report_rows)
    display(report_df)

    # Other化の中身（一覧）
    for col, items in other_mapping.items():
        if len(items) == 0:
            print(f"{col}: Other化なし")
        else:
            preview = items[:50]
            more = "" if len(items) <= 50 else f" ... (+{len(items)-50} more)"
            print(f"{col}: Other化 {len(items)}件 → {preview}{more}")

    if SAVE_OTHER_MAPPING and len(other_mapping) > 0:
        import json
        with open(OTHER_MAPPING_PATH, "w", encoding="utf-8") as f:
            json.dump(other_mapping, f, ensure_ascii=False, indent=2)
        print("Saved mapping:", OTHER_MAPPING_PATH)
else:
    print("Other化: OFF（必要なら OTHERIZE_ENABLED=True にして、OTHER_COLS に列名を入れてください）")


## 2) 目的変数・説明変数・型判定（叩き台）

- ここでは「数値→continuous / それ以外→nominal」を叩き台にします。
- 目的変数は `1/0` や `True/False` でも **分類（nominal）として扱う**想定。

In [ ]:
# === 2) target / predictors / variable_types (base) ===
# 目的変数をここで指定
target_col = "survived"
independent_cols = [c for c in df.columns if c != target_col]

# --- 除外（手動 + 自動） ---
excluded_reasons = {}

# 手動除外
for c in globals().get("MANUAL_EXCLUDE_COLS", []):
    if c in independent_cols:
        excluded_reasons[c] = "manual"

# 自動除外（IDっぽい列）
if globals().get("AUTO_EXCLUDE_ID_LIKE", False):
    patterns = [re.compile(p, re.IGNORECASE) for p in globals().get("AUTO_EXCLUDE_NAME_PATTERNS", [])]
    ratio_th = float(globals().get("AUTO_EXCLUDE_UNIQUE_RATIO_THRESHOLD", 0.98))
    min_nu   = int(globals().get("AUTO_EXCLUDE_MIN_NUNIQUE", 50))

    for c in list(independent_cols):
        if c in excluded_reasons:
            continue

        non_null = int(df[c].notna().sum())
        if non_null == 0:
            continue

        nunique = int(df[c].nunique(dropna=True))
        uniq_ratio = nunique / non_null

        name = str(c).lower()
        name_hit = any(p.search(name) for p in patterns)

        is_str = pd.api.types.is_string_dtype(df[c]) or (df[c].dtype == object)
        is_int = pd.api.types.is_integer_dtype(df[c])

        # 保守的ルール：
        # 1) 名前がIDっぽい + ほぼユニーク
        # 2) 文字列列でほぼユニーク（UUIDなど）
        # 3) 整数列でほぼユニーク（user_idがintなど）
        if nunique >= min_nu and uniq_ratio >= ratio_th:
            if name_hit:
                excluded_reasons[c] = f"auto:name_pattern + high_unique (nunique={nunique}, ratio={uniq_ratio:.3f})"
            elif is_str:
                excluded_reasons[c] = f"auto:string_high_cardinality (nunique={nunique}, ratio={uniq_ratio:.3f})"
            elif is_int:
                excluded_reasons[c] = f"auto:int_high_cardinality (nunique={nunique}, ratio={uniq_ratio:.3f})"

# 反映
excluded_cols = sorted(excluded_reasons.keys())
independent_cols = [c for c in independent_cols if c not in excluded_reasons]

# 表示
print(f"Predictors: {len(independent_cols)} (excluded {len(excluded_cols)})")
if excluded_cols:
    excl_rows = []
    for c in excluded_cols:
        non_null = int(df[c].notna().sum())
        nunique  = int(df[c].nunique(dropna=True))
        ratio    = (nunique / non_null) if non_null else float("nan")
        excl_rows.append({"col": c, "reason": excluded_reasons[c], "dtype": str(df[c].dtype), "nunique": nunique, "unique_ratio": ratio})
    excluded_df = pd.DataFrame(excl_rows).sort_values(["reason","col"]).reset_index(drop=True)
    display(excluded_df)
else:
    excluded_df = pd.DataFrame(columns=["col","reason","dtype","nunique","unique_ratio"])
    print("No predictors were excluded.")

# 叩き台：数値→continuous、それ以外→nominal
# ※ 目的変数は 1/0, True/False のときも “分類” として扱うのが安全
variable_types = {target_col: "nominal"}
for c in independent_cols:
    variable_types[c] = "continuous" if pd.api.types.is_numeric_dtype(df[c]) else "nominal"

# --- 表示：推定結果を一覧で確認（ここを見て手動overrideを決める） ---
summary = []
for c in [target_col] + independent_cols:
    nunique = int(df[c].nunique(dropna=True))
    summary.append({
        "col": c,
        "dtype": str(df[c].dtype),
        "nunique": nunique,
        "inferred_type": variable_types[c],
    })

vt_df = pd.DataFrame(summary).sort_values(["inferred_type","nunique","col"], ascending=[True, True, True])
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
vt_df.head(400)


## 3) 手動上書き（ここだけ編集すればOK）

例：
- 数値だけどコード（地域コード等）→ `nominal`
- 順序がある（ランク、ティア等）→ 実装が `ordinal` に対応している場合は `ordinal`（未対応なら `nominal` にしておく）
- 目的変数が `0/1` なら、可視化の並び替え用に `positive_class=1` を後で指定すると安定します。

In [ ]:
# === 3) overrides (edit here) ===
overrides = {
    # 例：
    # "segment_code": "nominal",
    # "tier": "nominal",   # ordinal未対応ならnominal
    # "age": "continuous",
}

variable_types.update(overrides)

# 上書き結果の確認（overridesが空ならエラーになるので分岐）
if len(overrides) == 0:
    print("overrides は空です（手動の上書きなし）。")
else:
    pd.DataFrame([{"col": k, "type": v} for k, v in overrides.items()]).sort_values("col").reset_index(drop=True)


## 4) 調整パラメータ（ここを触る）

「分岐数を増減させたい」「連続値の分割（ビン）を変えたい」などはここで調整します。

- **まず触るのは `max_depth` と `min_child_size`** が安全です（枝が細かすぎる問題に効く）。
- `alpha_split` / `alpha_merge` は木の構造を大きく変えるので、目的が“説明”か“予測”かで扱いを決めるのが良いです。
- `n_intervals`（連続値ビン数）は、見た目の閾値が不自然なときに試す価値があります。

> 注意：この実装（v6）の `CHAIDTree` が受け付ける引数は環境によって違う可能性があります。下の `run_tree()` は **受け付ける引数だけ自動で適用**し、適用されなかったものは一覧で表示します。


In [ ]:
# === 4) tuning knobs ===

# --- 学習（tree）側パラメータ ---
TREE_PARAMS = {
    "max_depth": 3,
    # ここから下は、実装が対応していれば使われます（未対応なら無視され、後で一覧表示されます）
    "min_parent_size": None,   # 例: 100
    "min_child_size": None,    # 例: 50
    "alpha_merge": None,       # 例: 0.05
    "alpha_split": None,       # 例: 0.05
    "n_intervals": None,       # 例: 10
    "interval_method": "quantile",   # 例: "equal_width" or "quantile"
    "max_intervals": None,     # 例: 64
    "include_missing_as_category": None,
    "allow_resplit": None,
}

# --- 可視化（graphviz）側パラメータ ---
VIZ_PARAMS = {
    "bar_mode": "image",
    "bar_orientation": "vertical",
    "bar_width_px": 14,
    "bar_height_px": 60,
    "image_dir": "chaid_viz_assets",
    # 表示の並び替え等（実装対応していれば）
    "sort_children_by": "auto",  # 連続は昇順、カテゴリはpositive割合で並べる…等
    "positive_class": 1,         # targetが 1/0 のときは 1 を指定すると安定（True/Falseなら True、yes/noなら "yes"）
}

# --- 出力設定 ---
OUTPUT_DIR = None   # 例: "out" にすると out/ 以下にまとまる（画像参照ズレを避けるため run_tree が chdir します）
OUTPUT_NAME = "chaid_tree"  # ファイル名（拡張子なし）


## 5) 実行（このセルだけ再実行すれば更新される）

In [ ]:
# === 5) run ===

def _filter_kwargs(func, kwargs):
    """funcが受け取れるkwargsだけ残す。残りはignoredとして返す。"""
    sig = inspect.signature(func)
    accepted = {}
    ignored = {}
    for k, v in kwargs.items():
        if v is None:
            continue
        if k in sig.parameters:
            accepted[k] = v
        else:
            ignored[k] = v
    return accepted, ignored

def run_tree(tree_params=None, viz_params=None, output_dir=OUTPUT_DIR, output_name=OUTPUT_NAME):
    tree_params = dict(TREE_PARAMS) if tree_params is None else dict(tree_params)
    viz_params  = dict(VIZ_PARAMS)  if viz_params  is None else dict(viz_params)

    # 1) CHAIDTree を作る（__init__が受け取れるものだけ渡す）
    init_kwargs, init_ignored = _filter_kwargs(CHAIDTree.__init__, tree_params)
    # __init__ には self があるので落とす
    init_kwargs.pop("self", None)

    tree = CHAIDTree(**init_kwargs)

    # 2) fit（fitが受け取れるものだけ渡す）
    fit_kwargs = dict(tree_params)
    # すでに __init__ に渡したものはここから除外しても良いが、対応実装差を考慮して一旦残しフィルタで落とす
    fit_kwargs, fit_ignored = _filter_kwargs(tree.fit, fit_kwargs)
    fit_kwargs.pop("self", None)

    tree.fit(
        df,
        target_col=target_col,
        independent_cols=independent_cols,
        variable_types=variable_types,
        **fit_kwargs,
    )

    # 3) to_graphviz（対応している可視化引数だけ渡す）
    viz_kwargs, viz_ignored = _filter_kwargs(tree.to_graphviz, viz_params)
    viz_kwargs.pop("self", None)

    # 4) 出力（画像参照ズレを避けるため、OUTPUT_DIR指定時は一時的にchdir）
    cwd = os.getcwd()
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        os.chdir(output_dir)

        # image_dir が相対パス想定なら、出力先内に作る
        img_dir = viz_kwargs.get("image_dir", "chaid_viz_assets")
        os.makedirs(img_dir, exist_ok=True)

        dot = tree.to_graphviz(**viz_kwargs)
        dot.render(filename=output_name, format="png", cleanup=True)
        display(Image(f"{output_name}.png"))

        os.chdir(cwd)
        saved_path = os.path.join(output_dir, f"{output_name}.png")
    else:
        # カレント直下に出す
        img_dir = viz_kwargs.get("image_dir", "chaid_viz_assets")
        os.makedirs(img_dir, exist_ok=True)

        dot = tree.to_graphviz(**viz_kwargs)
        dot.render(filename=output_name, format="png", cleanup=True)
        display(Image(f"{output_name}.png"))
        saved_path = f"{output_name}.png"

    # 5) 何が適用/無視されたか表示（重要：環境差の原因追跡）
    print("\n=== Applied / Ignored parameters ===")
    if init_ignored:
        print("[init] ignored:", init_ignored)
    if fit_ignored:
        print("[fit]  ignored:", fit_ignored)
    if viz_ignored:
        print("[viz]  ignored:", viz_ignored)
    print("Saved:", saved_path)

    return tree, dot

tree, dot = run_tree()


In [ ]:
import os, shutil

# 1) バー画像はプロジェクト直下に作る（dot も同じ場所で動かす）
assets_dir = "chaid_viz_assets"
os.makedirs(assets_dir, exist_ok=True)

dot = tree.to_graphviz(
    bar_mode="image",
    bar_orientation="vertical",
    image_dir=assets_dir,
    # positive_class=1,  # 1/0のときなど必要なら
)

# 2) まずカレントにPNGを作る（ここでエラーになりにくい）
dot.render(filename="chaid_tree", format="png", cleanup=True)

# 3) out/ に移動
timestamp = datetime.now().strftime("%y%m%d%H%M")

new_filename = f"chaid_tree_{timestamp}.png"
out_path = f"out/{new_filename}"

os.makedirs("out", exist_ok=True)
shutil.move("chaid_tree.png", out_path)

print("saved:", out_path)

## 6) 調整の例（よく使うやつだけ）

このセルは“テンプレ”です。必要な行だけコピペして `TREE_PARAMS` / `VIZ_PARAMS` を変えてください。

In [ ]:
# 例1) 枝が細かすぎる → まず min_child_size を上げる（実装が対応していれば効く）
# TREE_PARAMS["min_child_size"] = 100

# 例2) 深すぎる → 深さを浅くする
TREE_PARAMS["max_depth"] = 3

# 例3) 連続値の閾値が不自然 → ビン数や方法を変える（実装対応していれば）
# TREE_PARAMS["n_intervals"] = 10
# TREE_PARAMS["interval_method"] = "quantile"  # or "equal_width"

# 例4) 目的変数が True/False のとき
# VIZ_PARAMS["positive_class"] = True

# 例5) yes/no のとき（小文字/大文字はデータに合わせる）
# VIZ_PARAMS["positive_class"] = "yes"

# 反映したら、下を再実行
# tree, dot = run_tree()
